In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.model_selection import GridSearchCV

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from sklearn.ensemble import RandomForestClassifier

import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os

print(os.listdir('/content'))

In [ ]:
train = pd.read_csv('/content/customer_churn_dataset-training-master.csv')
test = pd.read_csv('/content/customer_churn_dataset-testing-master.csv')

print("Training Shape :", train.shape)
print("Testing Shape :", test.shape)

train.head()

In [ ]:
train.info()

train.describe()

train.isnull().sum()

In [ ]:
target_column = "Churn"

X_train = train.drop(target_column, axis=1)
y_train = train[target_column]

X_test = test.drop(target_column, axis=1)
y_test = test[target_column]

In [ ]:
num_cols = X_train.select_dtypes(include=np.number).columns
cat_cols = X_train.select_dtypes(exclude=np.number).columns

num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

X_train[num_cols] = num_imputer.fit_transform(X_train[num_cols])
X_test[num_cols] = num_imputer.transform(X_test[num_cols])

X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])
X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])

In [ ]:
encoders = {}

for col in cat_cols:
    le = LabelEncoder()

    combined = pd.concat([X_train[col], X_test[col]], axis=0)

    le.fit(combined)

    X_train[col] = le.transform(X_train[col])
    X_test[col] = le.transform(X_test[col])

    encoders[col] = le

In [ ]:
target_encoder = LabelEncoder()

y_train = target_encoder.fit_transform(y_train)
y_test = target_encoder.transform(y_test)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
baseline_model = RandomForestClassifier(
    random_state=42
)

baseline_model.fit(X_train_scaled, y_train)

baseline_pred = baseline_model.predict(X_test_scaled)

In [ ]:
print("Baseline Model Performance\n")

print("Accuracy :", accuracy_score(y_test, baseline_pred))
print("Precision:", precision_score(y_test, baseline_pred))
print("Recall   :", recall_score(y_test, baseline_pred))
print("F1 Score :", f1_score(y_test, baseline_pred))

print("\nClassification Report\n")
print(classification_report(y_test, baseline_pred))

In [ ]:
plt.figure(figsize=(5,4))

sns.heatmap(
    confusion_matrix(y_test, baseline_pred),
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title("Baseline Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
param_grid = {
    'n_estimators':[100,200,300],
    'max_depth':[5,10,20,None],
    'min_samples_split':[2,5,10],
    'min_samples_leaf':[1,2,4]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train_scaled, y_train)

print(grid.best_params_)

In [ ]:
best_model = grid.best_estimator_

best_model.fit(X_train_scaled, y_train)

optimized_pred = best_model.predict(X_test_scaled)

In [ ]:
print("Optimized Model Performance\n")

print("Accuracy :", accuracy_score(y_test, optimized_pred))
print("Precision:", precision_score(y_test, optimized_pred))
print("Recall   :", recall_score(y_test, optimized_pred))
print("F1 Score :", f1_score(y_test, optimized_pred))

print("\nClassification Report\n")
print(classification_report(y_test, optimized_pred))

In [ ]:
comparison = pd.DataFrame({

    "Metric":[
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ],

    "Baseline":[
        accuracy_score(y_test, baseline_pred),
        precision_score(y_test, baseline_pred),
        recall_score(y_test, baseline_pred),
        f1_score(y_test, baseline_pred)
    ],

    "Optimized":[
        accuracy_score(y_test, optimized_pred),
        precision_score(y_test, optimized_pred),
        recall_score(y_test, optimized_pred),
        f1_score(y_test, optimized_pred)
    ]

})

comparison

In [ ]:
comparison.set_index("Metric").plot(
    kind='bar',
    figsize=(8,5)
)

plt.title("Baseline vs Optimized Model")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.grid(axis='y')
plt.show()

In [ ]:
importance = pd.DataFrame({

    "Feature": X_train.columns,

    "Importance": best_model.feature_importances_

})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance.head(10)

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(
    data=importance.head(10),
    x="Importance",
    y="Feature"
)

plt.title("Top 10 Important Features")
plt.show()

In [ ]:
predictions = pd.DataFrame({

    "Actual": target_encoder.inverse_transform(y_test),

    "Predicted": target_encoder.inverse_transform(optimized_pred)

})

predictions.head()

In [ ]:
predictions.to_csv("customer_churn_predictions.csv", index=False)

print("Prediction file saved successfully!")

In [ ]:
import joblib

joblib.dump(best_model, "optimized_customer_churn_model.pkl")

print("Model Saved Successfully!")